# SCC0957 — Prática de Ciências de Dados II: Dados da Dengue PySuS
* Vinícius de Moraes - 13749910
* Rômulo Ferreira da Silva - 13734326
* João Pedro Barbosa Madeira - 13683038
* Pedro Silva dos Santos - 12688431
* Thiago Pasquotto Tavares - 15490194

## Carregando as bibliotecas

In [ ]:
!uv pip install pysus==1.0.1 -q
!uv pip install nbformat
!uv pip install plotly
!uv pip install matplotlib
!uv pip install geopandas

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import plotly.express as px
from pysus import SINAN
import polars as pl
import geopandas as gpd

## Carregando os dados

### 1. Carregando os Dados do SINAN

In [ ]:
sinan = SINAN().load() # Loads the files from DATASUS
files = sinan.get_files(dis_code=["DENG"])
parquet = sinan.download(files)

In [ ]:
df = pl.scan_parquet(
    [str(p) for p in parquet],
    extra_columns="ignore",
    missing_columns="insert"
)

In [ ]:
df.collect_schema()

### 2. Carregando os Dados Geográficos do Geodata

In [ ]:
url = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"

gdf = gpd.read_file(url)

if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")

# Calcula os centróides em um CRS projetado adequado ao território brasileiro
# e converte os pontos de volta para coordenadas geográficas usadas no mapa.
gdf_projetado = gdf.to_crs("EPSG:5880")
centroides = gdf_projetado.geometry.centroid.to_crs(gdf.crs)

gdf = gdf.assign(
    latitude=centroides.y,
    longitude=centroides.x,
)

gdf[["id", "name", "latitude", "longitude"]].head()

## Preparando os dados 

**Número de notificações/dia**

In [ ]:
result = (
    df
    .group_by("DT_NOTIFIC")
    .agg(pl.len().alias("Notificações"))
    .rename({"DT_NOTIFIC": "Dia"})
    .sort("Dia")
)

df_notificacoes = result.collect().to_pandas()

df_notificacoes['Dia'] = pd.to_datetime(df_notificacoes['Dia'], format='%Y%m%d', errors='coerce')

df_notificacoes = df_notificacoes.dropna(subset=['Dia'])

df_notificacoes

**Sazonalidade dos dados**

In [ ]:
df_notificacoes["Ano"] = df_notificacoes["Dia"].dt.year
df_notificacoes["Mês"] = df_notificacoes["Dia"].dt.month

df_sazonalidade = (
    df_notificacoes
    .groupby(["Ano", "Mês"])["Notificações"]
    .mean()
    .reset_index()
)

df_sazonalidade

**Classificação final da doença**

In [ ]:
classi_fin = (
    df
    .select("DENGUE")
    .collect()
    .to_pandas()
)

classi_fin = (
    df
    .select(
        pl.col("DENGUE")
        .str.strip_chars()
        .replace({
            "9": "Ignorado",
            "8": "Inconclusivo",
            "1": "Positivo",
            "2": "Negativo",
            "": "Ignorado"
        })
        .fill_null("Ignorado")
        .alias("DENGUE")
    )
    .collect()
    .to_pandas()
)

classi_fin

**Notificações por raça/etnia**

In [ ]:
cs_raca = (
    df
    .select(
        pl.col("CS_RACA")
        .str.strip_chars()
        .replace({
            '': 'Ignorado',
            '1': 'Branca',
            '2': 'Preta',
            '3': 'Amarela',
            '4': 'Parda',
            '5': 'Indígena',
            '9': 'Ignorado',
            '@': 'Ignorado'
        })
        .fill_null("Ignorado")
        .alias("CS_RACA")
    )
    .collect()
    .to_pandas()
)

cs_raca

**Evolução dos casos**

In [ ]:
df_evolucao = (
    df.select(
        pl.col('CON_EVOLUC')
        .str.strip_chars()
        .replace({
            '0': 'não se aplica',
            '1': 'cura',
            '2': 'óbito pelo agravo',
            '3': 'óbito por outras causas',
            '4': 'óbito em investigação',
            '9': 'ignorado',
            '': 'ignorado',
            ']': 'ignorado'
        })
        .fill_null('ignorado')
        .alias('CON_EVOLUC')
    )
    .collect()
    .to_pandas()
)

df_evolucao

**Número de notificações por estado de residência**

In [ ]:
df_estado = (
    df
    .with_columns(
        pl.col("SG_UF")
        .cast(pl.String)
        .str.strip_chars()
        .replace({
            "11": "RO",
            "12": "AC",
            "13": "AM",
            "14": "RR",
            "15": "PA",
            "16": "AP",
            "17": "TO",
            "21": "MA",
            "22": "PI",
            "23": "CE",
            "24": "RN",
            "25": "PB",
            "26": "PE",
            "27": "AL",
            "28": "SE",
            "29": "BA",
            "31": "MG",
            "32": "ES",
            "33": "RJ",
            "35": "SP",
            "41": "PR",
            "42": "SC",
            "43": "RS",
            "50": "MS",
            "51": "MT",
            "52": "GO",
            "53": "DF",
            "0": None,
            "": None,
            "`": None
        })
    )
    .drop_nulls("SG_UF")
    .group_by("SG_UF")
    .agg(pl.len().alias("Notificações"))
    .rename({"SG_UF": "Estado"})
    .sort("Estado")
    .collect()
    .to_pandas()
)

df_estado

**Número de notificações por município de residência**

A agregação territorial usa `ID_MN_RESI`; `ID_MUNICIP` identifica o município de notificação e é tratado separadamente na pipeline auditável.

In [ ]:
gdf["id_6"] = gdf["id"].astype(str).str[:6]

municipios_residencia = df.with_columns(
    pl.col("ID_MN_RESI")
    .cast(pl.String)
    .str.strip_chars()
    .str.replace(r"\.0$", "")
    .str.slice(0, 6)
    .alias("codigo_municipio_residencia")
)

df_municipio = (
    municipios_residencia
    .join(
        pl.from_pandas(
            gdf[["id_6", "name", "latitude", "longitude"]]
        ).lazy(),
        left_on="codigo_municipio_residencia",
        right_on="id_6",
        how="left",
    )
    .group_by(["name", "latitude", "longitude"])
    .agg(pl.len().alias("Notificações"))
    .drop_nulls("name")
    .collect()
    .to_pandas()
)

df_municipio.head()

## Visualizações

In [ ]:
media = df_notificacoes['Notificações'].mean()

fig = px.line(
    df_notificacoes,
    x='Dia',
    y='Notificações'
)

fig.add_hline(
    y=media,
    line_dash='dash',
    annotation_text=f"Média: {media:.0f}",
    annotation_position="top left"
)

fig.update_traces(line=dict(width=1))

fig.show()

In [ ]:
fig = px.line(
    df_sazonalidade,
    x="Mês",
    y="Notificações",
    color="Ano",
    markers=True,
    labels={
        "Mês": "Mês",
        "Notificações": "Notificações",
        "Ano": "Ano"
    },
    title="Sazonalidade das notificações por ano"
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(1, 13)),
        ticktext=[
            "Jan", "Fev", "Mar", "Abr",
            "Mai", "Jun", "Jul", "Ago",
            "Set", "Out", "Nov", "Dez"
        ]
    )
)

fig.show()

In [ ]:
from plotly.subplots import make_subplots

# Criar ano e mês
df_notificacoes["Ano"] = df_notificacoes["Dia"].dt.year
df_notificacoes["Mês"] = df_notificacoes["Dia"].dt.month

# Soma das notificações de cada mês de cada ano
df_mensal = (
    df_notificacoes
    .groupby(["Ano", "Mês"])["Notificações"]
    .sum()
    .reset_index()
)

nomes_meses = [
    "Janeiro", "Fevereiro", "Março", "Abril",
    "Maio", "Junho", "Julho", "Agosto",
    "Setembro", "Outubro", "Novembro", "Dezembro"
]

# Criar 12 gráficos
fig = make_subplots(
    rows=4,
    cols=3,
    subplot_titles=nomes_meses
)

for mes in range(1, 13):

    dados = df_mensal[df_mensal["Mês"] == mes]

    # Média histórica daquele mês
    media = dados["Notificações"].mean()

    linha = go.Scatter(
        x=dados["Ano"],
        y=dados["Notificações"],
        mode="lines+markers",
        name="Notificações",
        showlegend=False
    )

    media_linha = go.Scatter(
        x=dados["Ano"],
        y=[media] * len(dados),
        mode="lines",
        name="Média",
        line=dict(dash="dash"),
        showlegend=False
    )

    linha_idx = (mes - 1) // 3 + 1
    coluna_idx = (mes - 1) % 3 + 1

    fig.add_trace(
        linha,
        row=linha_idx,
        col=coluna_idx
    )

    fig.add_trace(
        media_linha,
        row=linha_idx,
        col=coluna_idx
    )

fig.update_layout(
    title="Tendência das notificações por mês ao longo dos anos",
    height=900,
    width=1200
)

fig.show()

In [ ]:
# Contagem dos resultados
dengue_counts = (
    classi_fin["DENGUE"]
    .value_counts()
    .reset_index()
)

dengue_counts.columns = ["Resultado", "Contagem"]

fig = px.bar(
    dengue_counts,
    x="Resultado",
    y="Contagem",
    text="Contagem",
    title="Número de casos de Dengue (2000–2023)",
    labels={
        "Contagem": "Número de casos",
        "Resultado": "Resultado"
    },
    color="Resultado"
)

fig.update_traces(textposition="outside")

fig.show()

In [ ]:
# Contagem dos resultados
raca_counts = (
    cs_raca["CS_RACA"]
    .value_counts()
    .reset_index()
)

raca_counts.columns = ["Raça", "Contagem"]

fig = px.bar(
    raca_counts,
    x="Raça",
    y="Contagem",
    text="Contagem",
    title="Número de casos de Dengue por Raça (2000–2023)",
    labels={
        "Contagem": "Número de casos",
        "Raça": "Raça"
    },
    color="Raça"
)

fig.update_traces(textposition="outside")

fig.show()

In [ ]:
# Contagem dos resultados
evolucao_counts = (
    df_evolucao["CON_EVOLUC"]
    .value_counts()
    .reset_index()
)

evolucao_counts.columns = ["Evolução", "Contagem"]

fig = px.bar(
    evolucao_counts,
    x="Evolução",
    y="Contagem",
    text="Contagem",
    title="Evolução dos casos de Dengue (2000–2023)",
    labels={
        "Contagem": "Número de casos",
        "Evolução": "Evolução"
    },
    color="Evolução"
)

fig.update_traces(textposition="outside")

fig.show()

In [ ]:
df_estado["Percentual"] = (
    df_estado["Notificações"]
    / df_estado["Notificações"].sum()
    * 100
)

df_estado = df_estado.sort_values(
    by="Notificações",
    ascending=False
)

fig = px.bar(
    df_estado,
    x="Estado",
    y="Notificações",
    color="Notificações",
    color_continuous_scale="Reds",
    text=df_estado["Percentual"].map(lambda x: f"{x:.1f}%"),
    title="Distribuição de Casos Positivos de Dengue por Estado (2000–2023)",
)

fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_title="Estado (UF)",
    yaxis_title="Número de Casos",
    showlegend=False
)

fig.show()

In [ ]:
fig = px.scatter_map(
    df_municipio,
    lat="latitude",
    lon="longitude",
    size="Notificações",
    hover_name="name",
    size_max=30,
    zoom=3,
    map_style="carto-positron",
    title="Número de notificações de Dengue por município de residência (2000–2023)",
    height=800,
)

fig.update_traces(marker={"color": "red"})

fig.show()

In [ ]:
top_mun = df_municipio.sort_values(
    'Notificações',
    ascending=False
).head(10)

fig = px.bar(
    top_mun.sort_values('Notificações'),
    x='Notificações',
    y='name',
    orientation='h',
    text='Notificações',
    title='Top 10 municípios de residência com mais notificações de Dengue (2000–2023)',
    labels={
        'Notificações': 'Número de notificações',
        'name': 'Município'
    },
    color='Notificações'
)

fig.update_traces(textposition='outside')

fig.show()